## Theme Analysis

In this notebook, I build a frequency × severity matrix to quantify what users say about Beli across themes. The goal is to surface which themes are most discussed and, more importantly, which Beli should prioritize when planning updates.

Before building the matrix, I reshape the data into a new table, `themes_long`. The `tagged_reviews` dataset stores all of a review's theme tags together in a single cell, which makes theme-level aggregation difficult. `themes_long` unpacks this into four columns — `review_id`, `source`, `theme`, `sentiment` — with one row per theme per review. A single review therefore appears in multiple rows if it touches multiple themes. This lets me aggregate at the theme level (counting theme mentions) rather than wrestling with multi-theme tags packed into one cell.

The frequency × severity matrix has six columns:
- `n_reviews` — number of reviews mentioning the theme
- `pct_of_reviews` — share of all reviews that touch the theme (frequency)
- `n_negative`/ `n_positive` — count of negative and positive mentions
- `pct_negative` — share of the theme's mentions that are negative (severity)
- `priority_score` — combines frequency and severity to rank themes

In [10]:
import pandas as pd

In [11]:
df = pd.read_csv('data/tagged_reviews.csv')
df.head()

,source,text,rating,version,date,title,storefront,author,review_id,theme_tags
0,app_store,"PLEASE bring back the old smaller pictures, th...",1,9.3.1,NaN,NEW UI SUCKS,us,wsonja112,0,"{""feature requests"": ""negative""}"
1,app_store,The UI is clunky and requiring invites to use ...,1,9.3.1,NaN,Poor design and scammy behavior,us,scknight4,1,"{""forced invite wall"": ""negative"", ""performanc..."
2,app_store,It is good at what it does and it’s free nd no...,5,9.3.1,NaN,Happy,us,Gstarbanana,2,"{""positive overall experience"": ""positive""}"
3,app_store,I have to share that I am drifting away from t...,2,9.3.1,NaN,Worse than Google Maps,us,brenda y. g.,3,"{""ranking and rating mechanics"": ""negative"", ""..."
4,app_store,All I want to do is find a restaurant near my ...,1,9.3.0,NaN,Useless,us,DestructoTex,4,"{""forced invite wall"": ""negative"", ""map and na..."


### Reshape into one row per theme

`tagged_reviews.csv` stores each review's themes in a single cell. I unpack that into `themes_long`, one row per (review, theme, sentiment), so I can aggregate by theme.

In [12]:
import json
import pandas as pd

df = pd.read_csv("data/tagged_reviews.csv")
total_reviews = len(df)

rows = []
for _, row in df.iterrows():
    tags = json.loads(row["theme_tags"])
    for theme, sentiment in tags.items():
        rows.append({
            "review_id": row.get("review_id"),
            "source": row.get("source"),
            "theme": theme,
            "sentiment": sentiment,
        })

themes_long = pd.DataFrame(rows)
print(f"{total_reviews} reviews, {len(themes_long)} theme mentions")
themes_long.head()

825 reviews, 1566 theme mentions


,review_id,source,theme,sentiment
0,0,app_store,feature requests,negative
1,1,app_store,forced invite wall,negative
2,1,app_store,performance and clunkiness,negative
3,1,app_store,feature requests,negative
4,2,app_store,positive overall experience,positive


### Frequency × severity matrix

For each theme: how many reviews mention it (frequency) and what share of those mentions are negative (severity). `priority_score` multiplies the two, so themes that are both common and mostly negative rank highest.

In [13]:
freq = themes_long.groupby("theme")["review_id"].nunique()
freq_pct = (freq / total_reviews * 100).round(1)

sent = themes_long.groupby(["theme", "sentiment"]).size().unstack(fill_value=0)
for col in ["negative", "neutral", "positive"]:
    if col not in sent:
        sent[col] = 0
sent["pct_negative"] = (sent["negative"] / sent[["negative", "neutral", "positive"]].sum(axis=1) * 100).round(1)

matrix = pd.DataFrame({
    "n_reviews": freq,
    "pct_of_reviews": freq_pct,
    "pct_negative": sent["pct_negative"],
    "n_negative": sent["negative"],
    "n_positive": sent["positive"],
})

matrix["priority_score"] = (matrix["n_reviews"] * matrix["pct_negative"] / 100).round(1)
matrix = matrix.sort_values("priority_score", ascending=False)
matrix

,n_reviews,pct_of_reviews,pct_negative,n_negative,n_positive,priority_score
theme,,,,,,
feature requests,307,37.2,66.6,227,100,204.5
forced invite wall,184,22.3,96.5,220,6,177.6
ranking and rating mechanics,218,26.4,56.3,130,96,122.7
performance and clunkiness,106,12.8,96.3,105,2,102.1
data privacy and account,76,9.2,95.1,77,3,72.3
map and navigation,100,12.1,64.7,66,35,64.7
positive overall experience,405,49.1,8.5,40,423,34.4


In [14]:
matrix.to_csv("data/theme_priority_matrix.csv")

top = matrix.index[0]
print(f"Top pain point: {top} — {int(matrix.loc[top, 'n_reviews'])} reviews "
      f"({matrix.loc[top, 'pct_of_reviews']}%), {matrix.loc[top, 'pct_negative']}% negative")

Top pain point: feature requests — 307 reviews (37.2%), 66.6% negative


### Commentary

From the analysis results, feature requests is the most-mentioned theme, accounting for 37.2% of total reviews, 66.6% of which carry negative sentiment. The forced invite wall also appears frequently, occupying 22.3% of reviews, with 96.5% of those carrying negative sentiment. Despite its lower priority score than feature requests, I argue the forced invite wall is the more important theme to investigate, because it is a single, well-defined feature, whereas feature requests are diffuse—more time-consuming to analyze and harder to weigh trade-offs across when deciding what to update. The invite wall's 96.5% negativity is near-unanimous and concentrated on one fixable surface, which is a stronger action signal than a slightly higher priority score spread across a vague bucket.


I also want to state a key limitation. The mean rating of users who left a written review is notably lower than Beli's overall rating on both the App Store and Google Play. This reflects voluntary response bias—a form of self-selection where users who opt to leave reviews skew toward stronger, often more negative, opinions than the silent majority. As a result, I can't claim these proportions reflect how all Beli users feel; the sample over-represents dissatisfied users.


Crucially, this limitation does not undercut the comparison driving my recommendation. The bias inflates negativity across every theme equally, since all themes are drawn from the same review pool—so it shifts absolute sentiment levels but not the ranking between themes. Even within this negatively-skewed sample, the invite wall's 96.5% negativity stands out sharply against other themes, including feature requests at 66.6%. So while I can't say what share of all users dislike the invite wall, I can say that among users who voice an opinion, opposition to it is near-unanimous and far more concentrated than for any other theme—which is what makes it actionable.